# 03. SFT 모델 평가 — 4조건 비교 + Langfuse


> **시나리오 안내** — 이 실습은 가상의 이커머스 마켓플레이스 **쇼핑몰**(자체 PB 라인: 베이직 · 프리미엄 · 라이트 · 한정판)를 소재로 합니다.

| 항목 | 내용 |
|------|------|
| 목적 | 02에서 학습한 SFT 모델의 "고유 기여도"를 4조건 비교로 정량 측정 |
| Base 모델 | Qwen/Qwen3.5-2B (FP16) |
| SFT 모델 | 02에서 HF Hub에 올린 merged FP16 |
| 추론 엔진 | HuggingFace Transformers (FP16) |
| 심사위원 | OpenAI GPT-4o-mini |
| 모니터링 | Langfuse Cloud (무료 플랜) |
| 런타임 | T4 GPU (Colab 무료) |
| 예상 소요 | ~25~35분 |
| 예상 비용 | ~$0.5 (120개 답변 평가) |

> **사전 조건**:
> 1. 02 노트북에서 `<user>/product-cs-qwen3.5-2b-sft` HF 모델 배포 완료 (private OK)
> 2. 01 노트북에서 `test.jsonl`이 Drive에 저장되어 있어야 함
> 3. Langfuse Cloud 계정 + Project + API Key 발급 (https://jp.cloud.langfuse.com)

---

### 왜 4조건으로 비교하는가?

가장 흔한 실수는 **"학습 전 답변 vs 학습 후 답변"** 두 개만 보고 "SFT 효과 있다!"라고 결론내는 것입니다. 이건 SFT를 **과대평가**합니다. 왜냐하면 **시스템 프롬프트만 잘 줘도** 비슷한 결과가 나오는 경우가 꽤 많기 때문이죠. "SFT 효과"라고 본 게 사실은 "프롬프트 효과"였을 수 있다는 겁니다.

그래서 **두 가지 변수(모델 종류 × 프롬프트 유무)를 직교(orthogonal)로 교차**시켜 4조건을 만듭니다.

**4조건 비교 프레임워크**:
- 조건 1: **Base + 시스템 프롬프트 없음** ← 출발선
- 조건 2: **Base + 시스템 프롬프트 있음** ← 프롬프트 효과만
- 조건 3: **SFT + 시스템 프롬프트 없음** ← SFT 효과만
- 조건 4: **SFT + 시스템 프롬프트 있음** ← 둘 다

이렇게 분리하면 각각의 기여도를 **수학적으로 떼어낼 수 있습니다**:

- **조건2 − 조건1** = 프롬프트만의 효과
- **조건3 − 조건1** = SFT만의 효과
- **조건3 − 조건2** = **SFT 고유 기여도** ⭐
- **조건4 − 조건3** = SFT 위에 프롬프트를 더 얹은 효과 (시너지)

결과별 분석:
- **조건2 ≈ 조건3** → 프롬프트만으로도 비슷한 수준
- **조건3 > 조건2** → SFT가 추가 가치를 만듦 ✅
- **조건4가 높더라도**, 조건2와 조건3을 함께 봐야 함

---
## Phase 1: 환경 설정

필요 패키지 설치 + 4가지 인증 정보 입력 (HF · OpenAI · Langfuse Public · Langfuse Secret).

In [1]:
# Unsloth가 Qwen3.5 호환 transformers(>=5.5.0)를 자동으로 맞춰 설치해줌
# → 02와 동일한 환경, KeyError: 'qwen3_5' 방지
!pip install --quiet -U unsloth unsloth_zoo
!pip install --quiet --no-deps "trl>=0.12" peft accelerate

# Qwen3.5 fast path 후보 라이브러리
!pip install --quiet -U flash-linear-attention

# Langfuse 최신 Python SDK 사용
!pip install --quiet -U openai langfuse tqdm

print("설치 완료 ✅")

### Langfuse API Key 발급 방법 (한 번만)

아래 `getpass()` 셀에서 `LANGFUSE_PUBLIC_KEY`와 `LANGFUSE_SECRET_KEY`를 입력받습니다. 아직 발급받지 않았다면 다음 순서로 한 번만 만들어두세요.

**1. 가입** — [https://jp.cloud.langfuse.com](https://jp.cloud.langfuse.com) 접속 → GitHub 또는 Google 계정으로 로그인 (**카드 등록 불필요**, 무료 플랜)

**2. Organization 만들기** — 첫 로그인 시 이름 입력 (예: `my-lecture`, 이미 만들었다면 건너뜀)

**3. Project 만들기** — Organization 진입 후 **"New Project"** 클릭 → 이름 입력 (예: `product-cs-sft-eval`)

**4. API Key 발급** — 좌측 하단 **Settings → API Keys → Create new API keys** 클릭
- 팝업에 **Public Key** (`pk-lf-...`)와 **Secret Key** (`sk-lf-...`) 두 줄이 한 번에 뜸
- ⚠️ **Secret Key는 이 화면에서만 확인 가능** — 창을 닫으면 다시 볼 수 없으니 바로 어딘가에 복사해두세요 (복사 후엔 재발급만 가능)

**5. 아래 셀 실행 시 차례로 붙여넣기** — `getpass`는 입력 글자를 가려주니 화면에 노출되지 않습니다.

> 💡 **왜 두 개인가?** `pk-lf-...`는 "어느 Project에 기록할지" 식별자, `sk-lf-...`는 "그 Project에 쓸 권한" 비밀번호. 내부적으로 둘이 `pk:sk` 한 쌍으로 묶여 HTTP Basic Auth 헤더가 됩니다. `sk`가 유출되면 그것만 재발급하면 돼요 (`pk`는 그대로 유지).

> 📊 **무료 플랜 한도**: 월 50,000 observations. 본 실습은 약 500 observations(120 Trace + 480 Score)이라 넉넉히 커버됩니다.

In [2]:
import os
from getpass import getpass
from huggingface_hub import login

# (1) HF 토큰 — 02에서 올린 SFT repo가 private이라면 read 권한 필수
hf_token = getpass("HF_TOKEN: ").strip()
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

# (2) OpenAI API Key — LLM-as-a-Judge용
os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ").strip()

# (3) Langfuse Cloud 키 (Japan 리전 프로젝트 기준)
#     https://jp.cloud.langfuse.com → Project Settings → API Keys
os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("LANGFUSE_PUBLIC_KEY (pk-lf-...): ").strip()
os.environ["LANGFUSE_SECRET_KEY"] = getpass("LANGFUSE_SECRET_KEY (sk-lf-...): ").strip()
os.environ["LANGFUSE_BASE_URL"] = "https://jp.cloud.langfuse.com"
os.environ["LANGFUSE_HOST"] = os.environ["LANGFUSE_BASE_URL"]

print("\n✅ 모든 인증 완료")

---
## Phase 2: Langfuse 초기화

이후 모든 답변 생성 + Judge 채점이 자동으로 Langfuse Cloud에 Trace + Score로 누적됩니다. 13강 개념의 실습 버전입니다.

In [3]:
import langfuse as _lf_mod
from langfuse import get_client

# 환경변수에서 키 자동 로드
# Jupyter에서 호스트/키를 바꿨다면 이 셀을 다시 실행해 새 클라이언트를 만듭니다.
langfuse = get_client()

print(f"✅ Langfuse 클라이언트 초기화 완료 (SDK v{_lf_mod.__version__})")
print(f"   대시보드: {os.environ['LANGFUSE_BASE_URL']}")

---
## Phase 3: Base 모델 + SFT 모델 로드

T4 16GB에 2B × 2 모델을 함께 올려 평가합니다. Qwen3.5 + Unsloth 조합에서는 환경에 따라 FP16 요청이 float32로 fallback될 수 있습니다.

In [4]:
import torch
from unsloth import FastLanguageModel
from huggingface_hub import whoami

# 02에서 올린 본인 계정의 SFT repo 자동 감지
HF_USERNAME = whoami()["name"]
SFT_REPO = f"{HF_USERNAME}/product-cs-qwen3.5-2b-sft"
print(f"SFT 모델 repo: {SFT_REPO}\n")

# --- Base 모델 (FP16) ---
# Unsloth가 내부적으로 Qwen3.5 패치 + transformers 호환 처리
print("[1/2] Base 모델 (Qwen/Qwen3.5-2B) 로드 중...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-2B",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=False,   # ★ 4bit OFF — FP16 그대로
)
FastLanguageModel.for_inference(base_model)   # KV cache ON, 학습용 메모리 해제
print(f"   ✅ VRAM 누적: {torch.cuda.memory_allocated() / 1e9:.2f} GB\n")

# --- SFT 모델 (FP16) ---
print("[2/2] SFT 모델 로드 중...")
sft_model, sft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_REPO,
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=False,
    token=True,           # private repo 대응
)
FastLanguageModel.for_inference(sft_model)
print(f"   ✅ VRAM 누적: {torch.cuda.memory_allocated() / 1e9:.2f} GB (T4 16GB 중)")

# Qwen3.5는 멀티모달 → 내부 text tokenizer 추출 (apply_chat_template / decode 안전)
base_tok = getattr(base_tokenizer, "tokenizer", base_tokenizer)
sft_tok  = getattr(sft_tokenizer,  "tokenizer", sft_tokenizer)

---
## Phase 4: 평가 데이터 준비

01 실습에서 만든 `test.jsonl`(74개)에서 **30개를 무작위 추출**해 평가용으로 사용합니다. 학습과 무관한 **OOD(Out-of-Distribution) 질문 5개**도 따로 준비해 Catastrophic Forgetting 체크용으로 씁니다.

**Catastrophic Forgetting (재앙적 망각)**: 기존에 잘 하던 능력을 잃어버림

> ⚠️ **test.jsonl은 절대 학습에 입력되지 않은 데이터** — 12강 Golden Dataset 개념의 실체입니다.

In [5]:
import json
import random
from pathlib import Path

random.seed(42)

# Drive 또는 로컬에서 test.jsonl 로드
LOCAL_PATH = Path("product-cs-sft-lab/sft-dataset/test.jsonl")
if LOCAL_PATH.exists():
    TEST_PATH = LOCAL_PATH
    print(f"📂 로컬 경로 사용: {TEST_PATH}")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    TEST_PATH = Path("/content/drive/MyDrive/product-cs-sft-lab/sft-dataset/test.jsonl")
    print(f"☁️  Drive 경로 사용: {TEST_PATH}")

assert TEST_PATH.exists(), f"test.jsonl을 찾을 수 없습니다: {TEST_PATH}"

test_data = [json.loads(l) for l in open(TEST_PATH, encoding="utf-8") if l.strip()]
print(f"✅ test.jsonl 로드: {len(test_data)}개\n")

# 30개 무작위 추출 (질문만 사용 — 정답 답변은 Judge가 알 필요 없음)
N_EVAL = min(30, len(test_data))
eval_samples = random.sample(test_data, N_EVAL)
eval_questions = [s["messages"][0]["content"] for s in eval_samples]

print(f"평가 질문 {len(eval_questions)}개 — 샘플 3개:")
for q in eval_questions[:3]:
    print(f"  - {q[:60]}{'...' if len(q) > 60 else ''}")

# Qwen3.5가 학습 전에 확실히 잘하던 5대 영역으로 구성
# → SFT 후 이 능력들이 유지되면 Catastrophic Forgetting 없음 확인 가능
ood_questions = [
    "파이썬으로 이진 탐색(binary search) 함수를 작성하고 각 줄에 주석을 달아줘.",
    "한 농부가 양을 20마리 가지고 있었습니다. 9마리를 제외하고 모두 죽었다면 몇 마리가 남았을까요?",
    "你好!请用中文告诉我,北京有哪些著名的旅游景点?至少列出3个。",
    "다음 문장을 자연스러운 영어로 번역해줘: '인공지능은 인류 역사상 가장 큰 혁명 중 하나다.'",
    "주기율표에서 원자번호 6번 원소의 이름, 기호, 그리고 주요 용도 3가지를 알려줘.",
]
print(f"\nCatastrophic Forgetting 확인 질문 {len(ood_questions)}개 — Qwen3.5 강점 영역으로 구성:")
print("  [1] 코드 생성 / [2] 수학·논리 / [3] 중국어 / [4] 번역 / [5] 과학 지식")
print("  → 이 5개가 모두 자연스럽게 답변되면 Catastrophic Forgetting 없음 ✅")

PHASE5_DRIVE_CACHE_PATH = None
if "/content/drive/" in str(TEST_PATH):
    PHASE5_DRIVE_CACHE_PATH = TEST_PATH.parent / "phase5_results.jsonl"
    print(f"💾 Phase 5 Drive 캐시 경로: {PHASE5_DRIVE_CACHE_PATH}")


---
## Phase 5: 4조건 답변 수집 + Langfuse Trace 기록

각 질문에 대해 **4개 조건 모두 답변을 받고**, 답변마다 Langfuse Trace를 만듭니다. 30 × 4 = **120개 Trace** 생성.

- 시스템 프롬프트는 "쇼핑몰 고객지원 상담원 톤"을 지시
- Trace의 `metadata.condition`과 `tags`로 어떤 조건인지 구분 → 대시보드에서 필터링 가능
- Langfuse에 `usage_details`(input/output/total tokens), `latency`, `estimated cost`도 함께 기록

In [6]:
import json
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from tqdm import tqdm
from langfuse import propagate_attributes

SYSTEM_PROMPT = (
    "당신은 쇼핑몰 고객 지원 고객지원 상담원입니다. "
    "고객 문의에 친절하고 공손한 어조로, 단계별로 안내해주세요. "
    "인사말로 시작하고, 마무리 멘트로 끝맺습니다."
)
MAX_NEW_TOKENS = 192
# 실습용 추정치: 로컬/Colab 추론은 provider bill이 없으므로 GPU 시간 기반 추정 비용을 사용합니다.
ESTIMATED_GPU_COST_PER_HOUR_USD = 0.35
PHASE5_CACHE_PATH = Path("phase5_results.jsonl")
CONDITIONS = [
    ("Base+NoPrompt", base_model, base_tok, "qwen3.5-2b-base", False),
    ("Base+Prompt",   base_model, base_tok, "qwen3.5-2b-base", True),
    ("SFT+NoPrompt",  sft_model,  sft_tok,  "qwen3.5-2b-sft",  False),
    ("SFT+Prompt",    sft_model,  sft_tok,  "qwen3.5-2b-sft",  True),
]
GROUPS = [CONDITIONS[:2], CONDITIONS[2:]]

def get_phase5_cache_paths():
    paths = [PHASE5_CACHE_PATH]
    drive_path = globals().get("PHASE5_DRIVE_CACHE_PATH")
    if drive_path:
        paths.append(Path(drive_path))
    return paths

def build_messages(question, with_system):
    return ([{"role": "system", "content": SYSTEM_PROMPT}] if with_system else []) + \
            [{"role": "user", "content": question}]

@torch.inference_mode()
def batch_generate(model, tok, requests, max_new_tokens=MAX_NEW_TOKENS):
    inputs = tok.apply_chat_template(
        [build_messages(q, s) for q, s in requests],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        padding=True,
    )
    prompt_lens = inputs["attention_mask"].sum(dim=1).tolist()
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    gen_started_at = datetime.now(timezone.utc)
    gen_start = perf_counter()
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=tok.eos_token_id,
    )
    gen_elapsed_s = perf_counter() - gen_start
    gen_finished_at = datetime.now(timezone.utc)
    output_lens = [(out[i].shape[0] - prompt_len) for i, prompt_len in enumerate(prompt_lens)]
    total_tokens = sum(p + o for p, o in zip(prompt_lens, output_lens)) or len(requests)
    total_cost_usd = (gen_elapsed_s / 3600.0) * ESTIMATED_GPU_COST_PER_HOUR_USD
    records = []
    for i, prompt_len in enumerate(prompt_lens):
        output_len = int(output_lens[i])
        item_total_tokens = int(prompt_len + output_len)
        share = (item_total_tokens / total_tokens) if total_tokens else (1 / len(requests))
        item_total_cost = total_cost_usd * share
        input_cost = item_total_cost * (prompt_len / item_total_tokens) if item_total_tokens else 0.0
        output_cost = item_total_cost * (output_len / item_total_tokens) if item_total_tokens else 0.0
        records.append({
            "response": tok.decode(out[i][prompt_len:], skip_special_tokens=True),
            "input_tokens": int(prompt_len),
            "output_tokens": output_len,
            "total_tokens": item_total_tokens,
            "latency_s": gen_elapsed_s,
            "estimated_cost_usd": item_total_cost,
            "input_cost_usd": input_cost,
            "output_cost_usd": output_cost,
            "started_at": gen_started_at,
            "finished_at": gen_finished_at,
        })
    return records

def save_phase5_results(results, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in results:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def load_phase5_results(path):
    path = Path(path)
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def run_phase5_eval(eval_questions, langfuse):
    results = []
    start = perf_counter()
    progress = tqdm(total=len(eval_questions) * len(CONDITIONS), desc="120개 답변 생성 중")
    print(f"Phase 5 설정 | max_new_tokens={MAX_NEW_TOKENS}, decode=greedy, per-model batch=2", flush=True)

    for q_idx, question in enumerate(eval_questions, start=1):
        for group in GROUPS:
            observations, reqs = [], []
            for cond_name, model, tok, model_label, with_sys in group:
                reqs.append((question, with_sys))
                observations.append((cond_name, model_label, with_sys))

            pending = []
            for cond_name, model_label, with_sys in observations:
                with propagate_attributes(tags=["sft-eval", cond_name]):
                    generation = langfuse.start_observation(
                        as_type="generation",
                        name="answer",
                        model=model_label,
                        input=build_messages(question, with_sys),
                        metadata={
                            "condition": cond_name,
                            "question": question,
                            "with_system_prompt": str(with_sys),
                            "model_type": "SFT" if cond_name.startswith("SFT") else "Base",
                            "cost_method": "estimated_gpu_time",
                            "latency_note": "same_batch_shared_latency",
                        },
                    )
                    pending.append((generation, cond_name, model_label, with_sys))

            generation_records = batch_generate(group[0][1], group[0][2], reqs)
            for (generation, cond_name, model_label, with_sys), record in zip(pending, generation_records):
                generation.update(
                    output=record["response"],
                    usage_details={
                        "input": record["input_tokens"],
                        "output": record["output_tokens"],
                        "total": record["total_tokens"],
                    },
                    cost_details={
                        "input": round(record["input_cost_usd"], 8),
                        "output": round(record["output_cost_usd"], 8),
                        "total": round(record["estimated_cost_usd"], 8),
                    },
                    metadata={
                        "condition": cond_name,
                        "question": question,
                        "with_system_prompt": str(with_sys),
                        "model_type": "SFT" if cond_name.startswith("SFT") else "Base",
                        "latency_s": round(record["latency_s"], 3),
                        "estimated_cost_usd": round(record["estimated_cost_usd"], 8),
                        "cost_method": "estimated_gpu_time",
                        "latency_note": "same_batch_shared_latency",
                    },
                )
                generation.end()
                results.append({
                    "question": question,
                    "condition": cond_name,
                    "response": record["response"],
                    "trace_id": generation.trace_id,
                    "observation_id": generation.id,
                    "latency_s": round(record["latency_s"], 3),
                    "input_tokens": record["input_tokens"],
                    "output_tokens": record["output_tokens"],
                    "total_tokens": record["total_tokens"],
                    "estimated_cost_usd": round(record["estimated_cost_usd"], 8),
                })
                progress.update(1)
                progress.set_postfix(question=q_idx, condition=cond_name, latency=f"{record['latency_s']:.1f}s")

        print(f"[progress] question {q_idx}/{len(eval_questions)} 완료 | 누적 answers={len(results)} | elapsed={(perf_counter()-start)/60:.1f}분", flush=True)

    progress.close()
    langfuse.flush()
    return results, perf_counter() - start


In [7]:
# Langfuse에 최신 trace/latency/cost를 다시 적재하려면 True로 바꾸세요.
FORCE_REGENERATE = True
cache_paths = get_phase5_cache_paths()
existing_cache = next((path for path in reversed(cache_paths) if Path(path).exists()), None)

if existing_cache and not FORCE_REGENERATE:
    results = load_phase5_results(existing_cache)
    elapsed_s = 0.0
    print(f"📂 Phase 5 캐시 로드: {existing_cache} ({len(results)}개)")
else:
    results, elapsed_s = run_phase5_eval(eval_questions, langfuse)
    for cache_path in cache_paths:
        save_phase5_results(results, cache_path)
        print(f"💾 Phase 5 결과 저장: {cache_path}")

print(f"\n✅ {len(results)}개 답변 생성 + Langfuse Trace 기록 완료")
if elapsed_s:
    print(f"총 소요 시간: {elapsed_s / 60:.2f}분")


---
## Phase 6: 자동 체크 (규칙 기반)

LLM 호출 없이 **즉시 측정 가능한 지표 4가지**를 집계합니다.

- `has_greeting`: 답변 앞부분에 "안녕하세요/죄송"
- `has_closing`: 답변 끝에 "문의/감사/도움" 마무리
- `response_length`: 평균 단어 수 — **품질 지표가 아니라 "학습 충실도" 지표**로 해석합니다 (아래 💡 참고)
- `formal_endings`: "습니다/세요/됩니다/십시오" 격식체 횟수


In [8]:
from collections import defaultdict

def auto_check(response: str) -> dict:
    # train.jsonl에 거의 없는 형식 요소(번호 리스트 등)는 제외하고,
    # 실제 데이터 분포에 분명히 존재하는 4개 지표만 본다.
    return {
        "has_greeting": any(w in response[:100] for w in ["안녕하세요", "죄송"]),
        "has_closing": any(w in response[-100:] for w in ["문의", "감사", "도움"]),
        "response_length": len(response.split()),
        "formal_endings": sum(response.count(e) for e in ["습니다", "세요", "됩니다", "십시오"]),
    }

agg = defaultdict(list)
for r in results:
    for k, v in auto_check(r["response"]).items():
        agg[(r["condition"], k)].append(v)

print(f"{'조건':<16}{'인사말%':>10}{'마무리%':>10}{'평균길이':>10}{'격식체':>10}")
print("-" * 60)
for cond_name, *_ in CONDITIONS:
    n = len(eval_questions)
    greet = sum(agg[(cond_name, 'has_greeting')]) / n * 100
    close = sum(agg[(cond_name, 'has_closing')]) / n * 100
    length = sum(agg[(cond_name, 'response_length')]) / n
    formal = sum(agg[(cond_name, 'formal_endings')]) / n
    print(f"{cond_name:<16}{greet:>9.0f}%{close:>9.0f}%{length:>10.1f}{formal:>10.1f}")

print("\n💡 조건1(Base+NoPrompt)과 조건3(SFT+NoPrompt)을 비교하세요.")
print("   같은 '시스템 프롬프트 없음'인데 형식 지표가 달라지면 → SFT 효과")


---
## Phase 7: 나란히 비교 출력 (1차 클라이맥스)

대표 3개 질문에 대해 4조건 답변을 한 화면에 펼쳐 봅니다. **시스템 프롬프트 없이도 SFT 모델이 행동을 바꾸는** 장면을 직접 눈으로 확인하는 단계입니다.

In [9]:
SAMPLE_N = 3
sample_questions = random.sample(eval_questions, SAMPLE_N)

for q in sample_questions:
    print("=" * 75)
    print(f"질문: {q}")
    print("=" * 75)
    for cond_name, *_ in CONDITIONS:
        ans = next(r["response"] for r in results
                   if r["question"] == q and r["condition"] == cond_name)
        print(f"\n[{cond_name}]")
        print(ans[:300] + ("..." if len(ans) > 300 else ""))
    print()

---
## Phase 8: LLM-as-a-Judge 평가 + Langfuse Score 기록

GPT-4o-mini가 각 답변을 **4차원(정확성·톤·구체성·환각)**으로 1~5점 채점합니다.

- 채점 결과를 해당 Trace에 **Score**로 첨부 → Langfuse 대시보드에서 조건별 분포 그래프 자동 생성
- 약 ~$0.5 비용, 5~10분 소요
- ★ 환각은 **낮을수록** 좋음 (지어낸 정보 없음 = 1, 심각하게 지어냄 = 5)

In [10]:
import json
from collections import defaultdict
from openai import OpenAI

oai = OpenAI()

JUDGE_SYSTEM = (
    "당신은 쇼핑몰 고객 지원 품질 평가 전문가입니다. "
    "고객 질문에 대한 모델 답변을 1~5점 척도로 채점하고 JSON으로만 출력하세요."
)

def call_judge(question: str, response: str) -> dict:
    user_prompt = f"""[고객 질문] {question}

[모델 답변]
{response}

다음 4개 항목을 1~5점으로 평가하세요:
- 정확성 (5=완벽, 1=완전히 틀림)
- 톤 (5=상담원 어조 완벽, 1=무뚝뚝)
- 구체성 (5=단계별 명확, 1=모호)
- 환각 (1=없음, 5=심각하게 지어냄, ★ 낮을수록 좋음)

JSON으로만 출력 (다른 말 절대 금지):
{{"정확성": N, "톤": N, "구체성": N, "환각": N}}"""

    res = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(res.choices[0].message.content)

scores_by_cond = defaultdict(lambda: defaultdict(list))

for r in tqdm(results, desc="LLM-as-a-Judge 채점 중"):
    try:
        scores = call_judge(r["question"], r["response"])
    except Exception as e:
        print(f"⚠️  채점 실패 ({r['condition']}): {e}")
        continue

    for metric, value in scores.items():
        try:
            value_f = float(value)
        except (TypeError, ValueError):
            continue

        langfuse.create_score(
            trace_id=r["trace_id"],
            observation_id=r.get("observation_id"),
            name=metric,
            value=value_f,
            comment=r["condition"],
            data_type="NUMERIC",
        )
        scores_by_cond[r["condition"]][metric].append(value_f)

langfuse.flush()
print("\n✅ Judge 채점 + Langfuse Score 기록 완료")


---
## Phase 9: Catastrophic Forgetting 체크 — Base vs SFT 나란히 비교

학습 데이터와 **무관한 질문 5개**(코드 생성, 수학 트릭, 중국어, 번역, 과학 지식)를 **Base와 SFT 두 모델에 같이 던져서** 답변을 비교합니다.

**판별 매트릭스**:

| Base 답변 | SFT 답변 | 진단 |
|---|---|---|
| 맞음 | 맞음 | ✅ 일반 능력 잘 보존됨 |
| 맞음 | 틀림 | ⚠️ **SFT가 추론 능력을 약화시킴** — 학습량/데이터 재설계 필요 |
| 틀림 | 틀림 | ℹ️ Base 원래의 한계 — SFT 탓 아님 |
| 틀림 | 맞음 | (드문 케이스 — SFT가 오히려 보강) |

In [11]:
print("=" * 75)
print("OOD 테스트 — Base vs SFT 나란히 비교")
print("  (같은 질문을 두 모델에 넣어서 SFT가 정말 능력을 잃었는지 판별)")
print("=" * 75)

# 긴 답변이 필요할 수 있는 OOD 질문(코드, 과학 지식 등)을 위해
# Phase 5 MAX_NEW_TOKENS보다 넉넉하게 — 상한 효과 제거
OOD_MAX_NEW_TOKENS = max(MAX_NEW_TOKENS, 512)

def truncate(s, n=250):
    return s[:n] + ("..." if len(s) > n else "")

for q in ood_questions:
    # Base와 SFT를 같은 조건(시스템 프롬프트 없음, 동일 max_new_tokens)으로 호출
    base_ans = batch_generate(base_model, base_tok, [(q, False)],
                              max_new_tokens=OOD_MAX_NEW_TOKENS)[0]["response"]
    sft_ans  = batch_generate(sft_model,  sft_tok,  [(q, False)],
                              max_new_tokens=OOD_MAX_NEW_TOKENS)[0]["response"]

    print(f"\n{'=' * 75}")
    print(f"[질문] {q}")
    print(f"{'-' * 75}")
    print(f"[Base 답변]")
    print(f"  {truncate(base_ans)}")
    print(f"{'-' * 75}")
    print(f"[SFT 답변]")
    print(f"  {truncate(sft_ans)}")

print("\n" + "=" * 75)
print("판단 기준:")
print("  ✅ 정상 (SFT 영향 없음):")
print("     - Base와 SFT 답변의 정확도가 비슷")
print("     - 둘 다 틀렸다면 → Base 원래의 한계 (SFT 탓 아님)")
print("     - 둘 다 맞췄다면 → 일반 능력이 잘 보존됨")
print("  ⚠️  SFT 손상 의심:")
print("     - Base는 맞췄는데 SFT만 틀림 → SFT가 추론/코드 능력을 약화시킨 신호")
print("     - 모든 SFT 답이 '안녕하세요! 쇼핑몰...' 톤으로 강제 → Catastrophic Forgetting")
print("=" * 75)


---
## Phase 10: Langfuse 대시보드 확인 (2차 클라이맥스)

이제 [jp.cloud.langfuse.com](https://jp.cloud.langfuse.com)에서 평가 결과를 시각화할 수 있습니다.

**확인할 것**:
1. **Traces 탭** → tag 필터 `sft-eval` → 120개 Trace
2. **Scores 탭** → 4조건별 정확성/톤/구체성/환각 점수 분포
3. **Tracing 컬럼** → Latency / Input Tokens / Output Tokens / Cost 확인
4. **조건2 vs 조건3**을 비교해 프롬프트 효과와 SFT 고유 기여도를 함께 해석

In [12]:
print("=" * 75)
print("📊 Langfuse 대시보드:")
print(f"   {os.environ['LANGFUSE_BASE_URL']}")
print("=" * 75)
print()
print("이번 실습에서 기록된 데이터:")
print(f"  • Traces: {len(results)}개 (4조건 x {len(eval_questions)}질문)")
print(f"  • Scores: 4지표 x {len(results)}개 ≈ {len(results) * 4}건")
print(f"  • Tag 필터: 'sft-eval'")
print()
print("대시보드에서 해볼 것:")
print("  1. Traces 페이지 → tag:sft-eval 필터 → metadata.condition으로 그룹화")
print("  2. Scores 페이지 → 조건별 평균 비교 (자동 그래프)")
print("  3. Tracing 컬럼 → Latency / Input Tokens / Output Tokens / Cost 확인")
print("  4. 조건 3 vs 조건 2 점수 차이 확인")
print("  5. 같은 질문에 4조건 답변 모두 보기")

---
## Phase 11: 최종 리포트 — SFT 고유 기여도 정량화

**공식**: SFT 고유 기여도 = (조건 3 점수) − (조건 2 점수)
                       = SFT만의 효과 − 프롬프트만의 효과

In [13]:
import statistics

print(f"{'조건':<16}{'정확성':>10}{'톤':>10}{'구체성':>10}{'환각(↓)':>12}")
print("-" * 60)
mean = {}
for cond_name, *_ in CONDITIONS:
    metrics = scores_by_cond[cond_name]
    mean[cond_name] = {
        m: statistics.mean(vals) if vals else 0.0
        for m, vals in metrics.items()
    }
    row = mean[cond_name]
    print(f"{cond_name:<16}"
          f"{row.get('정확성', 0):>10.2f}"
          f"{row.get('톤', 0):>10.2f}"
          f"{row.get('구체성', 0):>10.2f}"
          f"{row.get('환각', 0):>12.2f}")

print("\n" + "=" * 60)
print("SFT 고유 기여도 = (조건 3) - (조건 2)")
print("                  = SFT만의 효과 - 프롬프트만의 효과")
print("=" * 60)

base_p = mean.get("Base+Prompt", {})
sft_np = mean.get("SFT+NoPrompt", {})

for metric in ["정확성", "톤", "구체성"]:
    diff = sft_np.get(metric, 0) - base_p.get(metric, 0)
    sign = "+" if diff >= 0 else ""
    print(f"  {metric}: {sign}{diff:.2f}")

# 환각은 낮을수록 좋음 → Base − SFT (양수면 환각이 줄어든 것)
diff_h = base_p.get("환각", 0) - sft_np.get("환각", 0)
sign = "+" if diff_h >= 0 else ""
print(f"  환각 감소(↑좋음): {sign}{diff_h:.2f}")

print("\n💡 해석:")
print("  • 조건 3이 조건 2보다 높으면 SFT 고유 기여도가 있다고 볼 수 있음")
print("  • 차이가 작으면 프롬프트만으로도 비슷한 수준일 수 있음")
print("  • 조건 4가 높더라도 조건 2, 3을 함께 보세요")
print("  • 형식 일관성(Phase 6 자동 체크 표)도 함께 보세요")

---
## Phase 11-1: 시각화 — 조건 × 지표 라인 차트

Phase 11의 `scores_by_cond` 딕셔너리가 이미 **"조건 × 4지표"** 매트릭스이므로, 그대로 matplotlib로 그리면 **"평가기준 별 조건 비교 라인 차트"**가 나옵니다.

- X축: 4개 평가기준 (정확성·톤·구체성·환각)
- Y축: 평균 점수 (1~5)
- 4개 라인: 조건별(Base+NoPrompt / Base+Prompt / SFT+NoPrompt / SFT+Prompt)

> 💡 Langfuse Dashboard는 프로덕션 시계열 모니터링에 강하지만, 오프라인 평가의 "조건 × 지표" 2D 비교는 UI 제약이 있어서 matplotlib 한 셀이 훨씬 깔끔합니다.


In [14]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import statistics
import subprocess
from pathlib import Path

# --- Colab에서 한글 폰트 없으면 자동 설치 (1회만) ---
nanum_path = Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
if not nanum_path.exists():
    try:
        subprocess.run(["apt-get", "install", "-y", "-q", "fonts-nanum"], check=False,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception:
        pass
if nanum_path.exists():
    fm.fontManager.addfont(str(nanum_path))
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

# --- 데이터 준비 ---
metrics = ["정확성", "톤", "구체성", "환각"]
conditions = ["Base+NoPrompt", "Base+Prompt", "SFT+NoPrompt", "SFT+Prompt"]
colors = {
    "Base+NoPrompt": "#94a3b8",   # 회색
    "Base+Prompt":   "#3b82f6",   # 파랑
    "SFT+NoPrompt":  "#f59e0b",   # 주황
    "SFT+Prompt":    "#ef4444",   # 빨강
}

# --- 조건 × 지표 라인 차트 ---
fig, ax = plt.subplots(figsize=(11, 6))
x = list(range(len(metrics)))

for cond in conditions:
    y = []
    for m in metrics:
        vals = scores_by_cond.get(cond, {}).get(m, [])
        y.append(statistics.mean(vals) if vals else 0.0)
    ax.plot(x, y, marker="o", linewidth=2.5, markersize=10,
            label=cond, color=colors[cond])

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylabel("평균 점수 (1~5)", fontsize=12)
ax.set_title("평가기준 × 조건별 평균 점수 (환각은 낮을수록 좋음)", fontsize=13)
ax.legend(loc="lower left", fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_ylim(0, 5.2)

# 각 점에 수치 라벨
for cond in conditions:
    for i, m in enumerate(metrics):
        vals = scores_by_cond.get(cond, {}).get(m, [])
        if vals:
            v = statistics.mean(vals)
            ax.annotate(f"{v:.2f}", (i, v), textcoords="offset points",
                        xytext=(0, 8), ha="center", fontsize=9, color=colors[cond])

plt.tight_layout()
plt.show()

# --- 해석 힌트 ---
print("\n💡 해석 가이드:")
print("  • 같은 X 위치(같은 지표)에서 라인들의 높이 차이 → 조건별 점수 격차")
print("  • 'SFT+NoPrompt'가 'Base+Prompt'보다 위에 있어야 SFT 고유 기여도 있음")
print("  • '환각'은 낮을수록 좋음 (다른 지표와 반대 방향)")


---
## 완료

| 결과물 | 위치 |
|--------|------|
| 120개 Trace + ~480개 Score | Langfuse Cloud |
| 4조건 × 5지표 자동 체크 표 | Phase 6 출력 |
| SFT 고유 기여도 수치 | Phase 11 출력 |

**다음 실습**: `04_quantization.ipynb` — 동일 SFT FP16 모델을 GGUF로 양자화하고, **본 실습의 Langfuse Project에 양자화 버전 Score를 추가**해서 "FP16 → Q4_K_M" 품질 손실을 정량화합니다.

> ⚠️ **train.jsonl은 학습에만, test.jsonl은 평가에만** — 데이터 분리 원칙은 깨지 않습니다.